In [2]:
# General
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Machine Learning
import xgboost as xg

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error, roc_auc_score
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

from imblearn.over_sampling import SMOTE

# Feature Importance & Explainability
import shap

# Settings
import warnings
warnings.filterwarnings("ignore")

# Set random seed for reproducibility
SEED = 42
np.random.seed(SEED)

print("Libraries loaded. Ready to go!")

Libraries loaded. Ready to go!


In [3]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [4]:
train.head()

,id,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
0,0,37,70,36,Clayey,Sugarcane,36,4,5,28-28
1,1,27,69,65,Sandy,Millets,30,6,18,28-28
2,2,29,63,32,Sandy,Millets,24,12,16,17-17-17
3,3,35,62,54,Sandy,Barley,39,12,4,10-26-26
4,4,35,58,43,Red,Paddy,37,2,16,DAP


In [5]:
test.head()

,id,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous
0,750000,31,70,52,Sandy,Wheat,34,11,24
1,750001,27,62,45,Red,Sugarcane,30,14,15
2,750002,28,72,28,Clayey,Ground Nuts,14,15,4
3,750003,37,53,57,Black,Ground Nuts,18,17,36
4,750004,31,55,32,Red,Pulses,13,19,14


In [6]:
fertilizers = train['Fertilizer Name'].unique()
for i in fertilizers:
    print(i)

28-28
17-17-17
10-26-26
DAP
20-20
14-35-14
Urea


In [7]:
X = train.drop(columns=['Fertilizer Name'])
y = train['Fertilizer Name']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X = pd.get_dummies(X, columns=['Soil Type', 'Crop Type'])

X_train, x_val, y_train, y_val = train_test_split(X, y_encoded, test_size=0.2, random_state=SEED)

In [8]:
X_test = pd.read_csv('test.csv')

In [9]:
X_test = pd.get_dummies(X_test, columns=['Soil Type', 'Crop Type'])

In [17]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [5, 7, 9],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [100, 200],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_clf = xg.XGBClassifier(
    objective='multi:softprob',
    random_state=SEED,
    tree_method='hist',  
    use_label_encoder=False,
    eval_metric='mlogloss',
    device="cuda"
)

grid_search = GridSearchCV(
    estimator=xgb_clf,
    param_grid=param_grid,
    scoring='accuracy',
    cv=3,
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Fitting 3 folds for each of 48 candidates, totalling 144 fits
Best parameters: {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200, 'subsample': 0.8}
Best CV score: 0.19041166666666665


In [20]:
best_params_small = {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200, 'subsample': 0.8}
best_params_big = {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200, 'subsample': 0.8}


In [21]:
model = xg.XGBClassifier(**best_params_big)
model.fit(X, y_encoded)

proba = model.predict_proba(X_test)

top3_indices = np.argsort(proba, axis=1)[:, ::-1][:, :3]

top3_fertilizers = []
for preds in top3_indices:
    top3_fertilizers.append(' '.join(le.inverse_transform(preds)))

output = pd.DataFrame({
    'id': test.id,
    'Fertilizer Name': top3_fertilizers
})

output.to_csv('submission_copilot_xgb.csv', index=False)
print("Your submission with concatenated top 3 predictions was successfully saved!")

Your submission with concatenated top 3 predictions was successfully saved!


In [22]:
output.head()

,id,Fertilizer Name
0,750000,28-28 DAP 20-20
1,750001,17-17-17 20-20 10-26-26
2,750002,20-20 28-28 10-26-26
3,750003,14-35-14 17-17-17 20-20
4,750004,20-20 14-35-14 28-28


In [49]:
zz = pd.read_csv('train.csv')
y = zz['Fertilizer Name']

y = le.fit_transform(y)

In [59]:
def generate_stack(models, X, y, X_test, SEED=42, folds=5):
    n_models = len(models)
    meta_features_train = np.zeros((len(X), n_models))  
    meta_features_test = np.zeros((len(X_test), n_models))
    kf = KFold(n_splits=folds, shuffle=True, random_state=SEED)

    for i, model in enumerate(models):
        oof = np.zeros(len(X))
        preds = np.zeros(len(X_test))
        
        for train_idx, val_idx in kf.split(X):
            model.fit(X.iloc[train_idx], y.iloc[train_idx])
            oof[val_idx] = model.predict(X.iloc[val_idx])
            preds += model.predict(X_test) / folds
        
        meta_features_train[:, i] = oof
        meta_features_test[:, i] = preds

    meta_model = LogisticRegression()
    meta_model.fit(meta_features_train, y)
    return meta_model, meta_features_train, meta_features_test

models = [
    xg.XGBClassifier(**grid_search.best_params_, random_state=SEED, use_label_encoder=False, eval_metric='mlogloss'),
    RandomForestClassifier(random_state=SEED, n_jobs=-1),
    LogisticRegression(max_iter=1000, random_state=SEED)
]

if not isinstance(y, pd.Series):
    y = pd.Series(y)

meta_model, meta_features_train, meta_features_test = generate_stack(models, X, y, X_test)

stacked_preds = meta_model.predict(meta_features_train)

In [66]:
stacked_preds = meta_model.predict_proba(meta_features_test)

top3_indices = np.argsort(stacked_preds, axis=1)[:, ::-1][:, :3]

top3_fertilizers = []
for preds in top3_indices:
    top3_fertilizers.append(' '.join(le.inverse_transform(preds)))

output = pd.DataFrame({
    'id': test.id,
    'Fertilizer Name': top3_fertilizers
})

output.to_csv('submission_xgb_stack.csv', index=False)
print("Your submission with concatenated top 3 predictions was successfully saved!")

Your submission with concatenated top 3 predictions was successfully saved!


Optuna

In [63]:
!pip install optuna

  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.11.0
    Uninstalling typing_extensions-4.11.0:
      Successfully uninstalled typing_extensions-4.11.0


In [64]:
import optuna
import xgboost as xgb
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score
import numpy as np

def objective(trial):
    params = {
        'objective': 'multi:softprob',
        'eval_metric': 'mlogloss',
        'tree_method': 'hist',
        'use_label_encoder': False,
        'random_state': SEED,
        'device': 'cuda',
        'num_class': np.unique(y_train).shape[0],  # Needed for multi:softprob

        # Hyperparameters to tune
        'max_depth': trial.suggest_int('max_depth', 5, 9, step=2),
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.1),
        'n_estimators': trial.suggest_categorical('n_estimators', [100, 200]),
        'subsample': trial.suggest_categorical('subsample', [0.8, 1.0]),
        'colsample_bytree': trial.suggest_categorical('colsample_bytree', [0.8, 1.0]),
    }

    clf = xgb.XGBClassifier(**params)

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    scores = cross_val_score(clf, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)

    return scores.mean()

# Create study
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, timeout=600)  # 50 trials or 10 minutes

# Report results
print("Best parameters:", study.best_params)
print("Best CV score:", study.best_value)

[I 2025-06-08 16:11:47,308] A new study created in memory with name: no-name-a8d5d875-2ed3-4888-8a98-d3eebb0918e1
[I 2025-06-08 16:12:24,659] Trial 0 finished with value: 0.186285 and parameters: {'max_depth': 9, 'learning_rate': 0.09075906923373953, 'n_estimators': 100, 'subsample': 0.8, 'colsample_bytree': 1.0}. Best is trial 0 with value: 0.186285.
[I 2025-06-08 16:13:01,064] Trial 1 finished with value: 0.18795666666666666 and parameters: {'max_depth': 7, 'learning_rate': 0.089037066187756, 'n_estimators': 200, 'subsample': 1.0, 'colsample_bytree': 1.0}. Best is trial 1 with value: 0.18795666666666666.
[I 2025-06-08 16:14:05,298] Trial 2 finished with value: 0.18883666666666668 and parameters: {'max_depth': 9, 'learning_rate': 0.06871071366609513, 'n_estimators': 200, 'subsample': 1.0, 'colsample_bytree': 0.8}. Best is trial 2 with value: 0.18883666666666668.
[I 2025-06-08 16:14:27,253] Trial 3 finished with value: 0.18787666666666666 and parameters: {'max_depth': 5, 'learning_rate

Best parameters: {'max_depth': 9, 'learning_rate': 0.08171932178159289, 'n_estimators': 200, 'subsample': 0.8, 'colsample_bytree': 0.8}
Best CV score: 0.19065666666666667


In [65]:
model = xg.XGBClassifier(**study.best_params)
model.fit(X, y)

proba = model.predict_proba(X_test)

top3_indices = np.argsort(proba, axis=1)[:, ::-1][:, :3]

top3_fertilizers = []
for preds in top3_indices:
    top3_fertilizers.append(' '.join(le.inverse_transform(preds)))

output = pd.DataFrame({
    'id': test.id,
    'Fertilizer Name': top3_fertilizers
})

output.to_csv('submission_optuna.csv', index=False)
print("Your submission with concatenated top 3 predictions was successfully saved!")

Your submission with concatenated top 3 predictions was successfully saved!
